In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
tree_params = {"command": "uv", "args": ["run", "tree_server.py"]}
ports_params = {"command": "uv", "args": ["run", "ports_server.py"]}

async with MCPServerStdio(params=ports_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='list_listening_ports', title=None, description='List all ports currently listening for connections, with the owning process.', inputSchema={'properties': {}, 'title': 'list_listening_portsArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'list_listening_portsOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='port_info', title=None, description="\n    Show detailed info about a specific port — whether it's in use,\n    which process owns it, and any open connections.\n\n    Args:\n        port: The port number to inspect.\n    ", inputSchema={'properties': {'port': {'title': 'Port', 'type': 'integer'}}, 'required': ['port'], 'title': 'port_infoArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'port_infoOutput', 'type': 'object'}, icons=None, annotations=None, me

In [13]:
instructions = """You are a project assistant. You can inspect the project directory 
structure and manage ports on the system. Use the tree tool to explore directories 
and the port tools to list, check, scan, and kill ports as needed."""

request = "Show me the project directory tree and list 5 currently listening ports between 1000 and 10000"

model = "gpt-4.1-mini"

In [14]:
async with MCPServerStdio(params=tree_params, client_session_timeout_seconds=30) as tree_server:
    async with MCPServerStdio(params=ports_params, client_session_timeout_seconds=30) as ports_server:
        agent = Agent(
            name="system_assistant",
            instructions=instructions,
            model=model,
            mcp_servers=[tree_server, ports_server],
        )
        with trace("system_assistant"):
            result = await Runner.run(agent, request)
        display(Markdown(result.final_output))

The project directory tree at the top level and one level down is as follows:
```
/Users/olayinka/ai_bootcamp/agents/6_mcp/community_contributions/mrpeski
├── ports_server.py
├── tree_server.py
└── week_six_exercise.ipynb
```

Five currently listening ports between 1000 and 10000 are:
- Port 2276 (ibridge-mgmt)
- Port 3000 (hbci)
- Port 5000 (commplex-main)
- Port 7000 (afs3-fileserver)
- Port 7265 (unknown) 

If you need more details or want me to check any specific port or file, please let me know!